# TP 1 : la descente de gradient

Un problème dont on connaît la réponse exacte, pour pouvoir juger la descente au lieu de lui faire
confiance.

## 1.0 Un problème dont on connaît la réponse

442 patients diabétiques, dix mesures, et la progression de la maladie un an plus tard. Cette
fois les mesures sont dans leurs **unités d'origine** : l'âge en années (19 à 79), la tension
(62 à 133), le cholestérol total (97 à 301), et `s4` entre 2 et 9.

Le coût est la moyenne des carrés des erreurs,

$$J(w) = \frac{1}{m}\sum_{i=1}^{m}\big(w^T x^{(i)} - y^{(i)}\big)^2 ,$$

et il a une forme close, $w = (X^T X)^{-1}X^T y$. On s'en sert comme **réponse connue** : tout ce
que la descente produira sera comparé à elle.

**Q1.** La forme close $w = (X^T X)^{-1}X^T y$ ne se déduit que de ce coût-ci, la moyenne des
carrés. Des deux méthodes, la formule et la descente, laquelle vous restera-t-il quand le coût ne
sera plus celui-là ?

**Votre réponse :**



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

np.seterr(all="ignore")          # an exploding descent overflows; the numbers still tell the story
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

data = load_diabetes(as_frame=True, scaled=False)
feature_names = list(data.data.columns)
X, y = data.data.values, data.target.values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)


def design_matrix(X):
    return np.hstack([np.ones((len(X), 1)), X])


def cost(X, y, w):
    return np.mean((design_matrix(X) @ w - y) ** 2)


w_star = np.linalg.lstsq(design_matrix(X_train), y_train, rcond=None)[0]
print(f"coût minimal, par la forme close : {cost(X_train, y_train, w_star):.2f}")
print()
print(data.data.describe().loc[["min", "max", "std"]].round(1).to_string())

## 1.1 Le gradient, à la main

La descente part d'un $w$ quelconque et le corrige dans la direction qui fait baisser le coût :

$$\boxed{\ w \leftarrow w - \alpha \nabla_w J(w), \qquad
  \nabla_w J(w) = \frac{2}{m} X^T (Xw - y)\ }$$

Le gradient s'écrit en une ligne de numpy, mais une erreur de signe ou de facteur ne se voit pas :
la descente marche encore, en plus lent ou en divergeant. On l'écrit donc **deux fois** : une fois
sous sa forme matricielle, une fois comme la moyenne sur les $m$ patients dont elle est le
raccourci,

$$\frac{\partial J}{\partial w_j} = \frac{2}{m}\sum_{i=1}^{m}
  \big(w^T x^{(i)} - y^{(i)}\big)\, x_j^{(i)},$$

et on vérifie que les deux coïncident. Un facteur ou un signe oublié d'un seul côté se voit
aussitôt.

**Q2.** Dans le tableau des écarts-types affiché plus haut, `s1` vaut 34,6 et `s5` vaut 0,5 : les
valeurs de `s1` sont environ soixante-dix fois plus grandes. Le gradient multiplie chaque erreur par
la valeur de la colonne : laquelle des deux reçoit la plus grande composante de gradient ?
Qu'en concluez-vous pour un $\alpha$ unique appliqué aux deux ?

**Votre réponse :**



In [ ]:
def gradient(X, y, w):
    A = design_matrix(X)
    # TODO: return (2 / m) * A^T (A w - y)
    ...


def gradient_per_patient(X, y, w):
    """The same gradient, written as the mean over the m patients."""
    A = design_matrix(X)
    errors = A @ w - y
    # TODO: return the mean over the rows of 2 * error_i * x_i, which is a vector of n + 1 numbers
    ...


w_probe = np.full(X_train.shape[1] + 1, 0.5)
assert np.allclose(gradient(X_train, y_train, w_probe),
                   gradient_per_patient(X_train, y_train, w_probe))
assert np.allclose(gradient(X_train, y_train, w_star), 0.0, atol=1e-6)

print("les deux écritures du gradient coïncident")
print(f"norme du gradient au minimum : {np.linalg.norm(gradient(X_train, y_train, w_star)):.2e}")
print(f"norme du gradient en w = 0   : {np.linalg.norm(gradient(X_train, y_train, np.zeros(11))):.2e}")

**Q3.** La norme du gradient affichée passe de $8{,}28 \times 10^4$ en $w = 0$ à
$3{,}66 \times 10^{-10}$ au minimum. Quel test d'arrêt ces deux nombres suggèrent-ils pour la
descente ?

**Votre réponse :**



## 1.2 Chercher un pas qui marche

Un $\alpha$ trop petit met des siècles à arriver ; un $\alpha$ trop grand fait remonter le coût à
chaque pas et diverge. On balaie donc plusieurs ordres de grandeur, sur les mesures **en unités
d'origine**.

**Q4.** Le balayage qui suit essaie six valeurs de $\alpha$ sur les mesures en unités d'origine,
avec un seul $\alpha$ pour tous les poids, alors que `s1` va de 97 à 301 et `s4` de 2 à 9,1. Deux
issues possibles : l'une de ces valeurs atteint le minimum, ou bien chacune diverge ou avance trop
lentement pour y arriver. Laquelle attendez-vous ?

**Votre réponse :**



In [ ]:
def descend(X, y, alpha, n_steps):
    """Batch gradient descent from w = 0. Returns the weights and the cost after each step."""
    w = np.zeros(X.shape[1] + 1)
    history = []
    for _ in range(n_steps):
        # TODO: one gradient step, then append the new cost to history
        ...
        if not np.isfinite(w).all():
            break
    return w, np.array(history)


minimum = cost(X_train, y_train, w_star)
rows = []
for alpha in (1e-7, 1e-6, 5e-6, 1e-5, 1.3e-5, 2e-5):
    w_alpha, history = descend(X_train, y_train, alpha, 2000)
    rows.append({"α": alpha, "coût après 2 000 pas": history[-1],
                 "excès sur le minimum": history[-1] - minimum,
                 "diverge": not np.isfinite(history[-1])})

assert len(rows) == 6
print(f"minimum atteignable : {minimum:.2f}")
print(pd.DataFrame(rows).to_string(index=False, formatters={
    "α": "{:.1e}".format, "coût après 2 000 pas": "{:.1f}".format,
    "excès sur le minimum": "{:.1f}".format}))

fig, ax = plt.subplots(figsize=(7, 4.5))
for alpha in (1e-7, 1e-6, 5e-6, 1.3e-5):
    _, history = descend(X_train, y_train, alpha, 2000)
    ax.plot(history, label=f"α = {alpha:.1e}")
ax.axhline(minimum, color="crimson", ls="--", lw=1, label="minimum (forme close)")
ax.set_yscale("log")
ax.set_xlabel("pas")
ax.set_ylabel("coût J(w)")
ax.legend(fontsize=8)
plt.show()

**Q5.** Dans le tableau, $\alpha = 1{,}3 \times 10^{-5}$ laisse encore 404 d'excès sur le minimum
après deux mille pas, et l'$\alpha$ juste au-dessus, $2 \times 10^{-5}$, diverge. Entre « trop lent »
et « diverge » il n'y a donc rien à régler : qu'est-ce qui bloque, l'algorithme de descente ou les
unités des colonnes ?

**Votre réponse :**



## 1.3 Mettre les colonnes à la même échelle

Ce qui bloque n'est pas la descente, ce sont les **unités**. Un seul $\alpha$ s'applique à tous les
poids, alors que les colonnes vont de 0,5 d'écart-type pour `s5` à 34,6 pour `s1` : le pas qui
convient à l'une est soixante-dix fois trop grand ou trop petit pour l'autre.

La correction est d'enlever le problème plutôt que de régler autour : on ramène chaque colonne à une
moyenne nulle et un écart-type de 1. Les statistiques se calculent **sur l'entraînement seul**, puis
s'appliquent au test : une moyenne calculée sur le test ferait entrer le test dans le modèle.

Le modèle, lui, ne change pas : le minimum du coût reste le même, seule la route pour y arriver
change de forme. L'`assert` de la cellule le vérifie.

**Q6.** Après standardisation, toutes les colonnes auront le même écart-type. Un même $\alpha$
conviendra-t-il alors à tous les poids ?

**Votre réponse :**



In [ ]:
mean, deviation = X_train.mean(axis=0), X_train.std(axis=0)
# TODO: standardise both sets with the statistics of the training set
Z_train = ...
Z_test = ...

assert np.allclose(Z_train.mean(axis=0), 0.0)
assert np.allclose(Z_train.std(axis=0), 1.0)
assert not np.allclose(Z_test.mean(axis=0), 0.0)     # the test set is not recentred on itself

for name, M in (("unités d'origine", X_train), ("standardisé", Z_train)):
    print(f"{name:<18} écarts-types des colonnes : "
          f"de {M.std(axis=0).min():.3g} à {M.std(axis=0).max():.3g}")

w_star_z = np.linalg.lstsq(design_matrix(Z_train), y_train, rcond=None)[0]
assert np.isclose(cost(Z_train, y_train, w_star_z), cost(X_train, y_train, w_star))

rows = []
for alpha in (0.001, 0.01, 0.05, 0.1, 0.2, 0.24, 0.3):
    _, history = descend(Z_train, y_train, alpha, 2000)
    reached = np.flatnonzero(history < 1.01 * minimum)
    rows.append({"α": alpha, "coût après 2 000 pas": history[-1],
                 "pas pour arriver à 1 % du minimum": reached[0] + 1 if len(reached) else None})
print()
print(pd.DataFrame(rows).to_string(index=False, formatters={"coût après 2 000 pas": "{:.1f}".format}))

**Q7.** Dans la colonne `pas pour arriver à 1 % du minimum`, $\alpha = 0{,}2$ met 8 pas,
$\alpha = 0{,}24$ en met 203, et $\alpha = 0{,}3$ diverge. Que retenez-vous sur le choix d'un $\alpha$
juste en dessous de celui qui diverge ?

**Votre réponse :**



## 1.4 Ce que veut dire « jusqu'à convergence »

On s'arrête quand la norme du gradient passe sous un seuil $\varepsilon$, et un **plafond
d'itérations** sert de garde-fou : sans lui, un $\alpha$ mal réglé tourne indéfiniment.

**Q8.** Deux arrêts possibles : « le gradient est devenu petit » et « le coût ne baisse plus
beaucoup ». Lequel des deux un $\alpha$ minuscule trompe-t-il ?

**Votre réponse :**



In [ ]:
def descend_until(X, y, alpha, epsilon=1e-2, max_steps=100000):
    w = np.zeros(X.shape[1] + 1)
    for step in range(1, max_steps + 1):
        g = gradient(X, y, w)
        # TODO: stop and return (w, step, "gradient") as soon as the norm of g is below epsilon,
        # otherwise take the step
        ...
    return w, max_steps, "plafond"


rows = []
for alpha in (1e-5, 0.001, 0.05, 0.2):
    M = X_train if alpha == 1e-5 else Z_train
    w_end, steps, reason = descend_until(M, y_train, alpha)
    rows.append({"données": "origine" if alpha == 1e-5 else "standardisé", "α": alpha,
                 "arrêt": reason, "pas": steps, "coût atteint": cost(M, y_train, w_end),
                 "‖∇J‖ à l'arrêt": np.linalg.norm(gradient(M, y_train, w_end))})

assert rows[0]["arrêt"] == "plafond"
assert rows[-1]["arrêt"] == "gradient"
print(f"minimum : {minimum:.2f}")
print(pd.DataFrame(rows).to_string(index=False, formatters={
    "coût atteint": "{:.2f}".format, "‖∇J‖ à l'arrêt": "{:.3g}".format}))

**Q9.** La ligne `origine` s'est arrêtée après 100 000 pas sur un coût de 2 939,56, pour un minimum à
2 734,75. Quelle colonne du tableau permet de voir qu'elle n'a pas convergé, et pourquoi la fonction
doit-elle rendre cette information avec le modèle ?

**Votre réponse :**



## 1.5 Par lots, par mini-lots, ou un patient à la fois

Le gradient est une somme sur les $m$ patients. Trois façons de la consommer :

| | Exemples par pas | Mises à jour par époque |
|---|---|---|
| **par lots** | les $m$ | 1 |
| **mini-lots** | $b$ | $m / b$ |
| **stochastique** | 1 | $m$ |

Une **époque** est un passage complet sur les $m$ exemples : à budget d'époques égal, les trois
méthodes ne font donc pas le même nombre de mises à jour.

**Q10.** Le tableau donne les mises à jour par époque : 1 par lots, $m/b$ en mini-lots, $m$ en
stochastique. À $\alpha$ égal et à nombre d'époques égal, laquelle des trois avance le plus vite ?
Qu'en concluez-vous sur l'époque comme unité de comparaison ?

**Votre réponse :**



In [ ]:
def descend_batches(X, y, alpha, batch_size, n_epochs, decay=None, seed=0):
    """One pass per epoch, in shuffled batches. `decay`: alpha / (1 + updates / decay)."""
    rng = np.random.default_rng(seed)
    w = np.zeros(X.shape[1] + 1)
    history, updates = [], 0
    for _ in range(n_epochs):
        order = rng.permutation(len(X))
        for start in range(0, len(X), batch_size):
            rows = order[start:start + batch_size]
            step = alpha if decay is None else alpha / (1 + updates / decay)
            # TODO: one gradient step on the batch X[rows], y[rows], with step size `step`,
            # then count the update and append (updates, full training cost) to history
            ...
    return w, np.array(history)


alpha = 0.01
settings = [("par lots", len(Z_train)), ("mini-lots de 32", 32), ("stochastique", 1)]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
rows = []
for label, batch_size in settings:
    w_end, history = descend_batches(Z_train, y_train, alpha, batch_size, 50)
    ax.plot(history[:, 0], history[:, 1], label=label, lw=1)
    tail = history[-10:, 1]
    rows.append({"méthode": label, "exemples par pas": batch_size,
                 "mises à jour": int(history[-1, 0]), "coût final": history[-1, 1],
                 "oscillation en fin de course": tail.std()})

assert rows[0]["mises à jour"] == 50
assert rows[2]["mises à jour"] == 50 * len(Z_train)

w_decay, history_decay = descend_batches(Z_train, y_train, 0.05, 1, 50, decay=200)
ax.plot(history_decay[:, 0], history_decay[:, 1], label="stochastique, α décroissant", lw=1)
rows.append({"méthode": "stochastique, α décroissant", "exemples par pas": 1,
             "mises à jour": int(history_decay[-1, 0]), "coût final": history_decay[-1, 1],
             "oscillation en fin de course": history_decay[-10:, 1].std()})

ax.axhline(minimum, color="crimson", ls="--", lw=1, label="minimum (forme close)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("mises à jour")
ax.set_ylabel("coût sur tout l'entraînement")
ax.legend(fontsize=8)
plt.show()

print(f"minimum : {minimum:.2f}   (50 époques, α = {alpha} sauf mention)")
print(pd.DataFrame(rows).to_string(index=False, formatters={
    "coût final": "{:.1f}".format, "oscillation en fin de course": "{:.1f}".format}))

**Q11.** Les deux dernières lignes du tableau font le même nombre de mises à jour, 17 650, mais
l'`oscillation en fin de course` passe de 22,6 à 0,3 et le `coût final` de 2 958,5 à 2 738,3 quand
$\alpha$ décroît. Qu'apporte donc un $\alpha$ décroissant à la descente stochastique ?

**Votre réponse :**



**Q12.** Le coût de ce TP est **convexe** : une seule vallée, donc tout point où le gradient
s'annule est le minimum. Lequel de ces deux énoncés la convexité garantit-elle : « la descente
arrive au minimum », ou « si la descente arrive, c'est au minimum » ? La ligne arrêtée sur le
`plafond` en 1.4 vous donne la réponse.

**Votre réponse :**

